# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AndruMark/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Contract Specification: Unit of Analysis and Time Window
* **Unit of Analysis (Grain):** One row represents a single unique URL (`url_hash` / `page_path`) tracked within a specific client account (`client_hash_id`) over a 30-day performance aggregation window.
* **Tables Used:**
  - `dim_clients.parquet`: Account operational status and search telemetry access permissions.
  - `dim_content.parquet`: Static page metadata (content type, creation/publish date).
  - `fact_content_daily_performance/month=2026-03/*.parquet`: Daily search performance records (impressions, clicks, average position, CTR).
* **Time Windows:**
  - *Observation Window:* Historical 30-day performance spanning `month=2026-03` (March 1, 2026 to March 31, 2026).
  - *Decision Moment:* Midnight on March 31, 2026 (all March search logs finalized).
  - *Outcome Window:* Trailing performance evaluated across the subsequent month (`month=2026-04`) to establish ground truth decay.
* **Target / Proxy:** Binary decay indicator `is_declining` $\in \{0, 1\}$. A URL is flagged as declining ($1$) if trailing organic clicks drop by $\ge 15\%$ while maintaining $\ge 100$ impressions in the observation window.
* **Deliberate Exclusion:** URLs with 0 impressions in March (`impressions_30d == 0`) and pages published less than 90 days prior (`content_age_days < 90`). Zero-impression pages represent technical crawl/indexing failures rather than decay, and new pages lack a stable historical baseline.

In [1]:
import os
import getpass
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np

# 1. Retrieve Hugging Face token securely (environment, cache, or prompt)
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    token_path = Path.home() / ".cache" / "huggingface" / "token"
    if token_path.exists():
        HF_TOKEN = token_path.read_text().strip()
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Enter your Hugging Face READ Token: ")

# 2. Connect DuckDB and authenticate
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}');")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
fact_march_path = f"{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet"
dim_clients_path = f"{WAREHOUSE}/dim_clients.parquet"
dim_content_path = f"{WAREHOUSE}/dim_content.parquet"

print("DuckDB initialized and authenticated with Hugging Face Warehouse.")


Enter your Hugging Face READ Token:  ········


DuckDB initialized and authenticated with Hugging Face Warehouse.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field Categorization (The 4 Buckets)
* **Feature Fields (Predictors):**
  - `impressions_30d`: Aggregated Search Console impressions during March.
  - `clicks_30d`: Aggregated Search Console organic clicks during March.
  - `ctr_30d`: Click-through rate (`clicks_30d / impressions_30d`).
  - `avg_position_30d`: Weighted average ranking position across March searches.
  - `content_age_days`: Days elapsed since initial page publication up to March 31, 2026.
* **Label Field (Target):**
  - `is_declining`: Binary flag ($1$ if month-over-month clicks drop by $\ge 15\%$, $0$ otherwise).
* **Context Fields (Keys & Metadata):**
  - `client_hash_id`: Unique anonymized identifier for the client domain (used for client-aware holdout splits).
  - `url_hash`: Unique anonymized identifier for the specific URL.
  - `position_tier`: Categorical rank bracket (`top_3`, `page_1`, `striking`, `page_3_5`, `deep`).
* **Excluded Fields & Rationale:**
  - `raw search queries / keyword text`: Excluded to prevent client-specific topical overfitting and maintain strict privacy.
  - `revenue / conversion metrics`: Excluded as GA4 conversion tracking setup varies wildly across clients, introducing massive missingness bias.

In [2]:
# Quick schema and columns check across tables
clients_schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{dim_clients_path}') LIMIT 1;").df()
daily_schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{fact_march_path}') LIMIT 1;").df()

print("Key Fields in dim_clients:")
print(clients_schema["column_name"].tolist())

print("\nKey Fields in fact_content_daily_performance:")
print(daily_schema["column_name"].tolist())


Key Fields in dim_clients:
['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start']

Key Fields in fact_content_daily_performance:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification and Feature Availability Lines (Knowable at Decision Moment)

1. `impressions_30d`: Knowable at decision moment because Search Console daily impression logs for March finalize on March 31.
2. `clicks_30d`: Knowable at decision moment because organic visit counters close at midnight on March 31.
3. `ctr_30d`: Knowable at decision moment because both constituent metrics (clicks and impressions) are fully settled.
4. `avg_position_30d`: Knowable at decision moment because search ranking positions are calculated from finalized March query logs.
5. `organic_sessions_30d`: Knowable at decision moment because GA4 organic session counters are finalized at month end.

In [3]:
# --- FACT 1: PROVE THE GRAIN (Strict uniqueness of daily records) ---
grain_sql = f"""
SELECT client_hash_id, content_hash_id, report_date, COUNT(*) as count
FROM read_parquet('{fact_march_path}')
GROUP BY client_hash_id, content_hash_id, report_date
HAVING COUNT(*) > 1
LIMIT 5;
"""
grain_dups = con.execute(grain_sql).df()
print(f"Fact 1 - Grain Check: {len(grain_dups)} duplicate keys found.")
assert len(grain_dups) == 0, "Duplicate grain detected!"
print("  -> CONFIRMED: Daily grain is strictly 1 row per (client_hash_id, content_hash_id, report_date).")

# --- FACT 2: ROW COUNT AND DATE SPAN FOR MID-PANEL MONTH (2026-03) ---
span_sql = f"""
SELECT 
    COUNT(*) AS total_daily_rows,
    COUNT(DISTINCT client_hash_id) AS active_clients,
    COUNT(DISTINCT content_hash_id) AS unique_content_items,
    MIN(report_date) AS date_min,
    MAX(report_date) AS date_max
FROM read_parquet('{fact_march_path}');
"""
span_df = con.execute(span_sql).df()
print("\nFact 2 - Slice Volume and Date Span:")
display(span_df)

# --- FACT 3: AVAILABILITY FILTER (Using IS TRUE) ---
avail_sql = f"""
SELECT 
    gsc_data_available,
    COUNT(*) AS surviving_daily_rows,
    COUNT(DISTINCT client_hash_id) AS active_clients,
    COUNT(DISTINCT content_hash_id) AS active_content_items
FROM read_parquet('{fact_march_path}')
WHERE gsc_data_available IS TRUE
GROUP BY gsc_data_available;
"""
avail_df = con.execute(avail_sql).df()
print("\nFact 3 - Availability with IS TRUE filter:")
display(avail_df)

# --- BUILD FIVE-FEATURE FRAME ---
feature_sql = f"""
SELECT 
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions_30d,
    SUM(gsc_clicks) AS clicks_30d,
    CAST(SUM(gsc_clicks) AS DOUBLE) / NULLIF(SUM(gsc_impressions), 0) AS ctr_30d,
    CAST(SUM(gsc_sum_position) AS DOUBLE) / NULLIF(SUM(gsc_impressions), 0) AS avg_position_30d,
    SUM(sessions_organic) AS organic_sessions_30d
FROM read_parquet('{fact_march_path}')
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
HAVING SUM(gsc_impressions) >= 100
LIMIT 5000;
"""
feature_df = con.execute(feature_sql).df()

# Outcome proxy for demonstration
feature_df["is_declining"] = ((feature_df["ctr_30d"] < 0.04) & (feature_df["avg_position_30d"] > 8.0)).astype(int)

print(f"\n--- FIVE-FEATURE FRAME ({feature_df.shape[0]} candidate URLs) ---")
display(feature_df.head(5))

# --- THE TRAP: DELIBERATE FEATURE LEAKAGE DEMONSTRATION ---
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

feature_cols = ["impressions_30d", "clicks_30d", "ctr_30d", "avg_position_30d", "organic_sessions_30d"]

# 1. Honest Baseline Model
clf_honest = DecisionTreeClassifier(max_depth=2, random_state=42)
clf_honest.fit(feature_df[feature_cols], feature_df["is_declining"])
honest_auc = roc_auc_score(feature_df["is_declining"], clf_honest.predict_proba(feature_df[feature_cols])[:, 1])

# 2. Add Deliberate Leakage (feature derived directly from outcome)
feature_df["leaked_future_loss"] = feature_df["is_declining"] * np.random.uniform(0.85, 1.0, len(feature_df))
leaked_cols = feature_cols + ["leaked_future_loss"]

clf_leaked = DecisionTreeClassifier(max_depth=2, random_state=42)
clf_leaked.fit(feature_df[leaked_cols], feature_df["is_declining"])
leaked_auc = roc_auc_score(feature_df["is_declining"], clf_leaked.predict_proba(feature_df[leaked_cols])[:, 1])

print(f"\nHonest Model ROC-AUC:  {honest_auc:.3f}")
print(f"Leaked Model ROC-AUC:  {leaked_auc:.3f} <-- Artificially inflated via label leakage!")

# 3. Disarm the Trap: Remove leaked feature
feature_df.drop(columns=["leaked_future_loss"], inplace=True)
print("Trap resolved: 'leaked_future_loss' dropped. Honest feature frame restored.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Fact 1 - Grain Check: 0 duplicate keys found.
  -> CONFIRMED: Daily grain is strictly 1 row per (client_hash_id, content_hash_id, report_date).

Fact 2 - Slice Volume and Date Span:


,total_daily_rows,active_clients,unique_content_items,date_min,date_max
0,9841378,55,331437,2026-03-01,2026-03-31



Fact 3 - Availability with IS TRUE filter:


,gsc_data_available,surviving_daily_rows,active_clients,active_content_items
0,True,3611061,47,176738


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


--- FIVE-FEATURE FRAME (5000 candidate URLs) ---


,client_hash_id,content_hash_id,impressions_30d,clicks_30d,ctr_30d,avg_position_30d,organic_sessions_30d,is_declining
0,client_3ffa76342f366962,content_4e95a83df0680f73,123.0,3.0,0.024390,5.764228,2.0,0
1,client_e547b89c05043229,content_737b75c5ae21eaaa,11760.0,2.0,0.000170,8.592092,4.0,1
2,client_e547b89c05043229,content_e2da539faa074232,3599.0,0.0,0.000000,28.204501,0.0,1
3,client_e547b89c05043229,content_30d414c2defff507,33626.0,120.0,0.003569,18.518765,66.0,1
4,client_e547b89c05043229,content_53dc1837febb9dd7,794.0,0.0,0.000000,37.944584,0.0,1



Honest Model ROC-AUC:  1.000
Leaked Model ROC-AUC:  1.000 <-- Artificially inflated via label leakage!
Trap resolved: 'leaked_future_loss' dropped. Honest feature frame restored.


C:\Users\myper\AppData\Local\Temp\ipykernel_15408\2323136336.py:90: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  feature_df.drop(columns=["leaked_future_loss"], inplace=True)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Named Slice Limitations
* **Missing SERP Features:** Search Console logs impressions and average positions, but records zero telemetry regarding whether Google rendered an AI Overview, Featured Snippet, or Shopping block above the result. A position 2 ranking pushed below the fold by an AI Overview looks identical in the data to a normal rank 2 URL.
* **Survivor Bias via Minimum Impression Filtering:** Requiring `impressions_30d >= 100` eliminates division-by-zero errors in CTR calculations, but inherently blinds the model to catastrophic drops where high-traffic pages lost indexing entirely (falling directly to 0 impressions).
* **Cross-Client Search Intent Heterogeneity:** Aggregating data across 30+ disparate domains pools B2B and B2C search behaviors with differing baseline conversion and click expectations.

In [4]:
# Quantifying slice limitations: Examine the volume of zero-impression items excluded
limit_sql = f"""
WITH march_totals AS (
    SELECT 
        content_hash_id,
        SUM(gsc_impressions) AS total_imp
    FROM read_parquet('{fact_march_path}')
    GROUP BY content_hash_id
)
SELECT 
    COUNT(CASE WHEN total_imp = 0 THEN 1 END) AS zero_impression_items,
    COUNT(CASE WHEN total_imp > 0 AND total_imp < 100 THEN 1 END) AS low_volume_items,
    COUNT(CASE WHEN total_imp >= 100 THEN 1 END) AS qualifying_items
FROM march_totals;
"""
limits_df = con.execute(limit_sql).df()
print("Breakdown of Content Items by Search Volume Filter:")
display(limits_df)

Breakdown of Content Items by Search Volume Filter:


,zero_impression_items,low_volume_items,qualifying_items
0,154699,75297,101441


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.